<a href="https://colab.research.google.com/github/klozhechnikov/project_hse/blob/KL/calendar_parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install selenium

In [2]:
!apt-get update
!apt-get install -y chromium-chromedriver

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (1,996 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
chromium-chromedriver is already the newest versio

Как я понял, следующие две клетки кода нужны, чтобы версии силениума и хрома были совместимы.

In [6]:
# 1. Удаляем сломанные версии
!apt-get remove -y chromium-browser chromium-chromedriver
!apt-get autoremove -y

# 2. Добавляем официальный ключ и репозиторий Google
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'

# 3. Устанавливаем нормальный Google Chrome
!apt-get update
!apt-get install -y google-chrome-stable

# 4. Обновляем Selenium
!pip install -U selenium

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  apparmor libfuse3-3 snapd squashfs-tools systemd-hwe-hwdb udev
Use 'apt autoremove' to remove them.
The following packages will be REMOVED:
  chromium-browser chromium-chromedriver
0 upgraded, 0 newly installed, 2 to remove and 8 not upgraded.
After this operation, 243 kB disk space will be freed.
(Reading database ... 118715 files and directories currently installed.)
Removing chromium-chromedriver (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Removing chromium-browser (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages will be REMOVED:
  apparmor libfuse3-3 snapd squashfs-tools systemd-hwe-hwdb udev
0 u

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# Собираем настройки
options = Options()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080') # Задаем размер окна, чтобы элементы не съезжали

try:
    print("Пробуем запустить официальный Chrome...")
    # Selenium сам подтянет нужный драйвер
    driver = webdriver.Chrome(options=options)
    print("УРА! Браузер успешно запущен!")

    # Тестируем на КХЛ
    driver.get("https://www.khl.ru/")
    print(f"Заголовок сайта: {driver.title}")

    driver.quit()
except Exception as e:
    print(f"Ошибка: {e}")

Пробуем запустить официальный Chrome...
УРА! Браузер успешно запущен!
Заголовок сайта: 403


Видим что ошибка, значит надо скрываать что мы робот


In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')

# --- МАГИЯ МАСКИРОВКИ ОТ АНТИ-БОТОВ ---

# 1. Меняем User-Agent (говорим, что мы сидим с Windows 10)
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

# 2. Вырубаем флаги, которые кричат "Я РОБОТ!"
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

try:
    print("Пробуем зайти как человек...")
    driver = webdriver.Chrome(options=options)

    # 3. Дополнительно стираем свойства webdriver через JavaScript
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': '''
            Object.defineProperty(navigator, 'webdriver', {
              get: () => undefined
            })
        '''
    })

    driver.get("https://www.khl.ru/")
    print(f"Заголовок сайта: {driver.title}")

    driver.quit()
except Exception as e:
    print(f"Ошибка: {e}")

Пробуем зайти как человек...
Заголовок сайта: Хоккей – Континентальная Хоккейная Лига – Официальный сайт КХЛ


In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

class KHLCalendarParser:
    def __init__(self):
        options = Options()
        # Базовые настройки для Colab/Сервера
        options.add_argument('--headless=new')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')

        # --- МАГИЯ МАСКИРОВКИ ОТ АНТИ-БОТОВ ---
        options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
        options.add_argument('--disable-blink-features=AutomationControlled')
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)

        self.driver = webdriver.Chrome(options=options)

        # Скрываем флаг webdriver через JavaScript
        self.driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
            'source': '''
                Object.defineProperty(navigator, 'webdriver', {
                  get: () => undefined
                })
            '''
        })

    def parse_calendar(self, url):
        print(f"Открываем календарь: {url}")
        self.driver.get(url)

        # Ждем, пока прогрузится хотя бы одна карточка игры
        WebDriverWait(self.driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".card-game"))
        )

        games_data = []
        game_cards = self.driver.find_elements(By.CSS_SELECTOR, ".card-game")
        print(f"Найдено матчей на странице: {len(game_cards)}")

        for card in game_cards:
            try:
                team_1 = card.find_element(By.CSS_SELECTOR, ".card-game__club_left .card-game__club-name").text
                team_2 = card.find_element(By.CSS_SELECTOR, ".card-game__club_right .card-game__club-name").text

                try:
                    score_left = card.find_element(By.CSS_SELECTOR, ".card-game__center-score-left").text
                    score_right = card.find_element(By.CSS_SELECTOR, ".card-game__center-score-right").text
                    score = f"{score_left}:{score_right}"
                except:
                    score = "-:-"

                resume_link_element = card.find_element(By.CSS_SELECTOR, "a.card-game__hover-link[href*='resume']")
                match_link = resume_link_element.get_attribute("href")

                games_data.append({
                    "team_1": team_1,
                    "team_2": team_2,
                    "score": score,
                    "link": match_link
                })

            except Exception as e:
                continue

        df = pd.DataFrame(games_data)
        return df

    def close(self):
        self.driver.quit()

# Запуск
if __name__ == "__main__":
    calendar_url = "https://www.khl.ru/calendar/1369/00/"
    parser = KHLCalendarParser()

    try:
        df_matches = parser.parse_calendar(calendar_url)
        df_matches.to_csv("khl_calendar.csv", index=False, encoding="utf-8-sig")
        print("\nПарсинг завершен! Первые 5 строк:")
        print(df_matches.head())
    finally:
        parser.close()

Открываем календарь: https://www.khl.ru/calendar/1369/00/
Найдено матчей на странице: 752

Парсинг завершен! Первые 5 строк:
      team_1        team_2  score                                         link
0        СКА       Драконы    5:1  https://www.khl.ru/game/1369/897597/resume/
1     Сибирь         Барыс  2:1 Б  https://www.khl.ru/game/1369/898231/resume/
2  Динамо Мн  Автомобилист    8:2  https://www.khl.ru/game/1369/898232/resume/
3   Динамо М       Ак Барс    5:7  https://www.khl.ru/game/1369/898233/resume/
4    Трактор       Адмирал    1:0  https://www.khl.ru/game/1369/898234/resume/


In [10]:
df_matches

,team_1,team_2,score,link
0,СКА,Драконы,5:1,https://www.khl.ru/game/1369/897597/resume/
1,Сибирь,Барыс,2:1 Б,https://www.khl.ru/game/1369/898231/resume/
2,Динамо Мн,Автомобилист,8:2,https://www.khl.ru/game/1369/898232/resume/
3,Динамо М,Ак Барс,5:7,https://www.khl.ru/game/1369/898233/resume/
4,Трактор,Адмирал,1:0,https://www.khl.ru/game/1369/898234/resume/
...,...,...,...,...
747,Металлург Мг,Ак Барс,4:3 Б,https://www.khl.ru/game/1369/897492/resume/
748,Торпедо,Салават Юлаев,4:3,https://www.khl.ru/game/1369/897493/resume/
749,ЦСКА,Динамо М,6:2,https://www.khl.ru/game/1369/897494/resume/
750,Драконы,СКА,7:4,https://www.khl.ru/game/1369/897495/resume/
